# Assignment 2: The Dirty Ledger — work
All cleaning is done on a **copy**; `teaching_prices_messy.csv` is never overwritten. Data are invented teaching data (see `data_label`).

Name: Zeeshan Khan · ERP: 32133

## 1. Load and diagnose (before changing anything)

In [1]:
import pandas as pd, numpy as np
raw = pd.read_csv('teaching_prices_messy.csv')   # untouched original
df = raw.copy()                                   # working copy
print('shape:', df.shape)
print(df.dtypes)
print('\nmissing per column:'); print(df.isna().sum())

shape: (97, 6)
date          str
item          str
city          str
price         str
unit          str
data_label    str
dtype: object

missing per column:
date          0
item          0
city          0
price         1
unit          0
data_label    0
dtype: int64


In [2]:
# Problem 1: price is text, not numbers -> find non-numeric strings
bad_type = df[pd.to_numeric(df['price'], errors='coerce').isna() & df['price'].notna()]
print('price strings that do not parse as numbers:'); print(bad_type[['date','item','city','price']].to_string())
# Problem 2: missing price
print('\nmissing price rows:'); print(df[df['price'].isna()][['date','item','city','price']].to_string())

price strings that do not parse as numbers:
          date      item    city  price
17  2026-01-15  Staple A  City 2  1,250

missing price rows:
         date     item    city price
5  2026-01-01  Pulse C  City 2   NaN


In [3]:
# Problem 3: duplicate rows
print('exact duplicate rows:', df.duplicated().sum())
print(df[df.duplicated(keep=False)][['date','item','city','price']].to_string())
# Problem 4: inconsistent labels
print('\nitem labels:'); print(df['item'].value_counts())

exact duplicate rows: 1
          date      item    city price
8   2026-01-08  Staple A  City 1  99.5
96  2026-01-08  Staple A  City 1  99.5

item labels:
item
Staple A       24
Oil B          24
Pulse C        24
Sweetener D    24
oil b           1
Name: count, dtype: int64


In [4]:
# Problem 5: impossible / extreme prices -> compare each price with its own item-city median
num = pd.to_numeric(df['price'].str.replace(',', ''), errors='coerce')
df['_num'] = num
df['_item_key'] = df['item'].str.strip().str.title()
df['_ratio'] = df['_num'] / df.groupby(['_item_key','city'])['_num'].transform('median')
print(df[(df['_ratio']<0.7)|(df['_ratio']>1.4)][['date','item','city','price','_ratio']].round(2).to_string())
print('\nprice describe by item (after stripping commas):'); print(df.groupby('_item_key')['_num'].describe().round(1).to_string())

          date         item    city  price  _ratio
17  2026-01-15     Staple A  City 2  1,250   10.82
30  2026-01-22  Sweetener D  City 1   9999   60.33
40  2026-02-05        oil b  City 1  108.5    0.34

price describe by item (after stripping commas):
             count   mean     std    min    25%    50%    75%     max
_item_key                                                            
Oil B         25.0  316.2    45.5  108.5  311.0  324.0  337.0   350.0
Pulse C       23.0  205.9     3.9  200.0  202.5  206.0  208.5   213.5
Staple A      24.0  158.3   232.6   99.5  106.2  112.5  116.1  1250.0
Sweetener D   24.0  576.8  2006.9  149.0  161.0  167.8  174.9  9999.0


In [5]:
# Problem 6: the lower-case 'oil b' row - is it really Oil B?
print(df[df['date']=='2026-02-05'][['item','city','price']].to_string())
print('\nStaple A City 1 rows on that date:', ((df['item']=='Staple A')&(df['city']=='City 1')&(df['date']=='2026-02-05')).sum())
print('Oil B  City 1 rows on that date (case-insensitive):', ((df['item'].str.lower()=='oil b')&(df['city']=='City 1')&(df['date']=='2026-02-05')).sum())
sa = df[(df['item']=='Staple A')&(df['city']=='City 1')]['_num']
print('Staple A City 1 price range:', sa.min(), '-', sa.max(), '| Oil B City 1 (excluding that row) range:',
      df[(df['item']=='Oil B')&(df['city']=='City 1')]['_num'].min(), '-', df[(df['item']=='Oil B')&(df['city']=='City 1')]['_num'].max())

           item    city  price
40        oil b  City 1  108.5
41     Staple A  City 2  114.5
42        Oil B  City 1  318.0
43        Oil B  City 2  324.0
44      Pulse C  City 1  202.5
45      Pulse C  City 2  208.5
46  Sweetener D  City 1  161.5
47  Sweetener D  City 2  167.5

Staple A City 1 rows on that date: 0
Oil B  City 1 rows on that date (case-insensitive): 2
Staple A City 1 price range: 99.5 - 116.0 | Oil B City 1 (excluding that row) range: 298.0 - 344.0


In [6]:
# Coverage: how many rows per item and city, and per date?
print(pd.crosstab(df['item'], df['city']))
print('\nrows per date:'); print(df['date'].value_counts().sort_index().to_string())

city         City 1  City 2
item                       
Oil B            12      12
Pulse C          12      12
Staple A         12      12
Sweetener D      12      12
oil b             1       0

rows per date:
date
2026-01-01    8
2026-01-08    9
2026-01-15    8
2026-01-22    8
2026-01-29    8
2026-02-05    8
2026-02-12    8
2026-02-19    8
2026-02-26    8
2026-03-05    8
2026-03-12    8
2026-03-19    8


### Diagnosis summary (code that revealed each problem)
| # | Problem | Where | Revealed by |
|---|---|---|---|
| 1 | Price stored as text; one value has a thousands comma (`"1,250"`) | Staple A, City 2, 2026-01-15 | `dtypes`, `pd.to_numeric(errors='coerce')` |
| 2 | Missing price | Pulse C, City 2, 2026-01-01 | `isna().sum()` |
| 3 | Exact duplicate row | Staple A, City 1, 2026-01-08 (row 96) | `duplicated()` |
| 4 | Inconsistent label `oil b` | 2026-02-05, City 1 | `value_counts()` |
| 5 | Sentinel / impossible prices: `9999` (and `1,250` once parsed) | Sweetener D City 1, 2026-01-22; Staple A City 2, 2026-01-15 | `describe()`, ratio to item-city median |
| 6 | The `oil b` row is really **Staple A**: price 108.5 fits Staple A City 1 (99.5–116), Staple A City 1 has no row on that date, and Oil B City 1 already has one (318.0) | 2026-02-05 | `crosstab`, `groupby` counts |

## 2. Cleaning Code Court — each candidate line tried on a copy
The candidate lines are the standard one-liners an automated cleaner typically suggests. Each is run on its own on a copy of the baseline (price converted to numbers). *Valid observations* = the 93 rows that are genuine, correctly priced observations (every row except the 3 invalid prices and the duplicate).

In [7]:
base = raw.copy()
base['price'] = pd.to_numeric(base['price'].str.replace(',', ''), errors='coerce')   # baseline: text -> number
invalid_idx = {5, 17, 30, 96}                      # missing, 1,250, 9999, duplicate row
valid_idx = set(base.index) - invalid_idx          # 93 genuine observations

def trial(name, func):
    w = base.copy()
    out = func(w)
    rows = len(out)
    missing = out['price'].isna().sum() if 'price' in out else None
    keydup = out.duplicated(['date','item','city']).sum() if {'date','item','city'}.issubset(out.columns) else None
    lost = len(valid_idx - set(out.index))
    kept = [i for i in out.index if i in valid_idx]
    changed = int((out.loc[kept,'price'] != base.loc[kept,'price']).sum()) if 'price' in out else None
    labels = int((out['item'] != base.loc[out.index,'item']).sum()) if 'item' in out else None
    return dict(line=name, rows=rows, missing_price=missing, key_dups=keydup, valid_obs_lost=lost, valid_prices_changed=changed, labels_changed=labels, max_price=out['price'].max() if 'price' in out else None)

print('BASELINE rows', len(base), '| missing price', base['price'].isna().sum(), '| key duplicates', base.duplicated(['date','item','city']).sum(), '| max price', base['price'].max())
trials = [
 ('L1  df.drop_duplicates()',                         lambda d: d.drop_duplicates()),
 ("L2  df['item']=df['item'].str.strip().str.title()", lambda d: d.assign(item=d['item'].str.strip().str.title())),
 ('L3  df.dropna()',                                   lambda d: d.dropna()),
 ("L4  df['price'].fillna(df['price'].mean())",        lambda d: d.assign(price=d['price'].fillna(d['price'].mean()))),
 ("L5  df[df['price']<1000]",                          lambda d: d[d['price']<1000]),
 ("L6  df[df['price']!=9999]",                         lambda d: d[d['price']!=9999]),
 ("L7  df['price']=df['price'].round(0)",              lambda d: d.assign(price=d['price'].round(0))),
 ("L8  df.drop(columns=['unit','data_label'])",        lambda d: d.drop(columns=['unit','data_label'])),
 ("L9  pooled IQR filter on price",                    lambda d: d[(d['price']>=d['price'].quantile(.25)-1.5*(d['price'].quantile(.75)-d['price'].quantile(.25)))&(d['price']<=d['price'].quantile(.75)+1.5*(d['price'].quantile(.75)-d['price'].quantile(.25)))|d['price'].isna()]),
 ("L10 df['price'].clip(upper=df['price'].quantile(.99))", lambda d: d.assign(price=d['price'].clip(upper=d['price'].quantile(.99)))),
]
res = pd.DataFrame([trial(n,f) for n,f in trials]); print(res.to_string(index=False))

BASELINE rows 97 | missing price 1 | key duplicates 1 | max price 9999.0
                                                 line  rows  missing_price  key_dups  valid_obs_lost  valid_prices_changed  labels_changed  max_price
                             L1  df.drop_duplicates()    96              1         0               0                     0               0    9999.00
    L2  df['item']=df['item'].str.strip().str.title()    97              1         2               0                     0               1    9999.00
                                      L3  df.dropna()    96              0         1               0                     0               0    9999.00
           L4  df['price'].fillna(df['price'].mean())    97              0         1               0                     0               0    9999.00
                             L5  df[df['price']<1000]    94              0         1               0                     0               0     350.00
                           

In [8]:
# Detail for the lines that need a closer look
w = base.copy()
t = w.assign(item=w['item'].str.strip().str.title())
print('L2 effect on the mislabelled row (index 40):', base.loc[40,'item'], '->', t.loc[40,'item'])
print('L2 creates', t.duplicated(['date','item','city']).sum(), 'duplicate date-item-city key(s):'); print(t[t.duplicated(['date','item','city'],keep=False)][['date','item','city','price']].to_string())
print('\nL3 removes:'); print(base.loc[sorted(set(base.index)-set(base.dropna().index)),['date','item','city','price']].to_string())
m = base['price'].mean(); print('\nL4 would fill the missing Pulse C price (true level ~200-213) with the pooled mean =', round(m,1))
print('\nL5 / L6 detail:')
print(' L5 removes', sorted(set(base.index)-set(base[base.price<1000].index)), '-> the 1,250 row, the 9999 row AND the missing-price row (NaN<1000 is False), so 3 rows vanish silently; the mislabelled row is not caught')
print(' L6 removes', sorted(set(base.index)-set(base[base.price!=9999].index)), '-> only the 9999 row; max price afterwards =', base[base.price!=9999]['price'].max(), '(1,250 survives)')
r = base[(base.index.isin(valid_idx))&(base['price']!=base['price'].round(0))]
print('\nL7 changes', len(r), 'valid half-rupee prices, e.g.'); print(r[['date','item','city','price']].head(4).to_string())
print('\nL9 pooled fences remove:'); 
q1,q3=base.price.quantile([.25,.75]); lo,hi=q1-1.5*(q3-q1),q3+1.5*(q3-q1); print(' fences', round(lo,1), round(hi,1), '| rows outside:', ((base.price<lo)|(base.price>hi)).sum(), '| of which valid:', len([i for i in base[(base.price<lo)|(base.price>hi)].index if i in valid_idx]))
print(' It removes only the 1,250 and 9999 rows here, but the fences (about -100 to 547) are pooled across items, so a wrong Staple A price of e.g. 300 would pass; it also cannot see the mislabelled row.')
c = base.price.clip(upper=base.price.quantile(.99))
print('\nL10 clip: 9999 becomes', round(c[30],2), '(a number that never existed); 1,250 stays', c[17], '; valid values changed:', int((c.loc[sorted(valid_idx)]!=base.loc[sorted(valid_idx),'price']).sum()))

L2 effect on the mislabelled row (index 40): oil b -> Oil B
L2 creates 2 duplicate date-item-city key(s):
          date      item    city  price
8   2026-01-08  Staple A  City 1   99.5
40  2026-02-05     Oil B  City 1  108.5
42  2026-02-05     Oil B  City 1  318.0
96  2026-01-08  Staple A  City 1   99.5

L3 removes:
         date     item    city  price
5  2026-01-01  Pulse C  City 2    NaN

L4 would fill the missing Pulse C price (true level ~200-213) with the pooled mean = 315.5

L5 / L6 detail:
 L5 removes [5, 17, 30] -> the 1,250 row, the 9999 row AND the missing-price row (NaN<1000 is False), so 3 rows vanish silently; the mislabelled row is not caught
 L6 removes [30] -> only the 9999 row; max price afterwards = 1250.0 (1,250 survives)

L7 changes 35 valid half-rupee prices, e.g.
          date      item    city  price
8   2026-01-08  Staple A  City 1   99.5
9   2026-01-08  Staple A  City 2  105.5
12  2026-01-08   Pulse C  City 1  202.5
13  2026-01-08   Pulse C  City 2  208.5

L

## 3. Final cleaning (my decisions) — on a copy

In [9]:
clean = raw.copy()
clean['price_raw'] = clean['price']                                  # keep the original text for traceability
clean['flag'] = ''

# (a) exact duplicate row -> drop (verified identical on every column; first copy kept)
n0 = len(clean); clean = clean.drop_duplicates(keep='first').copy()
print('duplicate rows dropped:', n0-len(clean))

# (b) label: 'oil b' on 2026-02-05 City 1 is Staple A (evidence above) -> correct and flag
mask = (clean['item']=='oil b') & (clean['date']=='2026-02-05') & (clean['city']=='City 1')
assert mask.sum()==1
clean.loc[mask,'item'] = 'Staple A'
clean.loc[mask,'flag'] = 'label corrected: oil b -> Staple A (Oil B City 1 already present; price fits Staple A)'
clean['item'] = clean['item'].str.strip()                           # whitespace-only safety, changes nothing here

# (c) price text -> number (remove thousands comma)
clean['price'] = pd.to_numeric(clean['price'].str.replace(',', ''), errors='coerce')

# (d) invalid prices -> set to missing, keep the row, flag it (no imputation)
m1 = clean['price'].isna()
clean.loc[m1,'flag'] = 'price missing in source - left missing'
m2 = (clean['item']=='Staple A') & (clean['city']=='City 2') & (clean['date']=='2026-01-15')
clean.loc[m2,'price'] = np.nan; clean.loc[m2,'flag'] = 'price "1,250" implausible (neighbours 105-116) - set to missing'
m3 = clean['price']==9999
clean.loc[m3,'price'] = np.nan; clean.loc[m3,'flag'] = 'price 9999 is a sentinel - set to missing'

# (e) types, order
clean['date'] = pd.to_datetime(clean['date'])
clean = clean.sort_values(['date','item','city']).reset_index(drop=True)
print(clean[clean['flag']!=''][['date','item','city','price_raw','price','flag']].to_string())

duplicate rows dropped: 1
         date         item    city price_raw  price                                                                                    flag
3  2026-01-01      Pulse C  City 2       NaN    NaN                                                  price missing in source - left missing
21 2026-01-15     Staple A  City 2     1,250    NaN                         price "1,250" implausible (neighbours 105-116) - set to missing
30 2026-01-22  Sweetener D  City 1      9999    NaN                                               price 9999 is a sentinel - set to missing
44 2026-02-05     Staple A  City 1     108.5  108.5  label corrected: oil b -> Staple A (Oil B City 1 already present; price fits Staple A)


## 4. Before / after

In [10]:
def summary(d, pricecol):
    p = pd.to_numeric(d[pricecol].astype(str).str.replace(',', ''), errors='coerce')
    return {'rows': len(d), 'distinct items': d['item'].str.strip().str.title().nunique() if pricecol=='price' and d is raw else d['item'].nunique(),
            'distinct dates': d['date'].nunique(), 'exact duplicate rows': int(d.duplicated().sum()),
            'missing price': int(p.isna().sum()), 'price min': p.min(), 'price max': p.max(), 'price mean': round(p.mean(),2), 'price median': p.median()}
b = summary(raw, 'price'); b['distinct items'] = raw['item'].nunique()
a = summary(clean, 'price')
ba = pd.DataFrame({'before (raw)': b, 'after (clean)': a}); print(ba.to_string())
print('\nprice describe() before (text parsed, as-is):'); print(pd.to_numeric(raw['price'].str.replace(',', ''), errors='coerce').describe().round(2).to_string())
print('\nprice describe() after:'); print(clean['price'].describe().round(2).to_string())
print('\nmissing counts per column after:'); print(clean.isna().sum().to_string())

                      before (raw)  after (clean)
rows                         97.00          96.00
distinct items                5.00           4.00
distinct dates               12.00          12.00
exact duplicate rows          1.00           0.00
missing price                 1.00           3.00
price min                    99.50          99.50
price max                  9999.00         350.00
price mean                  315.46         203.61
price median                200.00         200.00

price describe() before (text parsed, as-is):
count      96.00
mean      315.46
std      1007.56
min        99.50
25%       142.25
50%       200.00
75%       304.00
max      9999.00

price describe() after:
count     93.00
mean     203.61
std       79.92
min       99.50
25%      149.00
50%      200.00
75%      298.00
max      350.00

missing counts per column after:
date          0
item          0
city          0
price         3
unit          0
data_label    0
price_raw     1
flag          0


## 5. Does every item appear the expected number of times?

In [11]:
print(pd.crosstab(clean['item'], clean['city']))
assert (pd.crosstab(clean['item'], clean['city'])==12).all().all(), 'each item-city should appear 12 times'
assert clean['date'].nunique()==12 and (clean.groupby('date').size()==8).all(), 'each date should have 8 rows'
assert not clean.duplicated(['date','item','city']).any(), 'date-item-city must be unique'
assert set(clean['item'])=={'Staple A','Oil B','Pulse C','Sweetener D'}
print('\nAll expectation checks passed: 4 items x 2 cities x 12 dates = 96 rows, each key once.')
print('Valid priced observations:', int(clean['price'].notna().sum()), '| flagged rows:', int((clean['flag']!='').sum()))


city         City 1  City 2
item                       
Oil B            12      12
Pulse C          12      12
Staple A         12      12
Sweetener D      12      12

All expectation checks passed: 4 items x 2 cities x 12 dates = 96 rows, each key once.
Valid priced observations: 93 | flagged rows: 4


In [12]:
chk = raw.drop(index=96).copy()
chk['p'] = pd.to_numeric(chk['price'].str.replace(',',''), errors='coerce')
chk.loc[40,'item'] = 'Staple A'
chk['date'] = pd.to_datetime(chk['date'])
mm = chk.merge(clean[['date','item','city','price']], on=['date','item','city'], how='outer', indicator=True, suffixes=('_raw','_clean'))
print(mm['_merge'].value_counts().to_string())
diff = mm[(mm['p']!=mm['price_clean']) & mm['price_clean'].notna()]
print('valid prices that differ between raw and clean:', len(diff))
print('prices set to missing (expected 3):', int(mm['price_clean'].isna().sum()))

_merge
both          96
left_only      0
right_only     0
valid prices that differ between raw and clean: 0
prices set to missing (expected 3): 3


## 6. Save the cleaned data
The cleaned table is saved next to this notebook as `teaching_prices_cleaned.csv`. The columns `price_raw` and `flag` keep the audit trail.

In [13]:
clean.to_csv('teaching_prices_cleaned.csv', index=False)
print(pd.read_csv('teaching_prices_cleaned.csv').shape, 'saved -> teaching_prices_cleaned.csv')

(96, 8) saved -> teaching_prices_cleaned.csv


## 7. Extra cross-check (after the decisions above were made)
The Week-1 `teaching_prices_clean.csv` is the tidy version of the same ledger. It is used here only as an independent check on my decisions, not to make them.

In [14]:
ref = pd.read_csv('teaching_prices_clean_week1_reference.csv', parse_dates=['date'])
x = clean.merge(ref[['date','item','city','price']], on=['date','item','city'], how='outer', suffixes=('_mine','_ref'), indicator=True)
print(x['_merge'].value_counts().to_string())
both = x[x['price_mine'].notna()]
print('priced rows that equal the reference:', int((both['price_mine']==both['price_ref']).sum()), 'of', len(both))
print('\nthe 3 rows I left missing, with the reference value (not applied):')
print(x[x['price_mine'].isna()][['date','item','city','price_ref']].to_string(index=False))
g = x.pivot_table(index=['date','item'], columns='city', values='price_mine'); print('\nCity 2 - City 1 gap in the cleaned data (all complete pairs):', [float(v) for v in sorted((g['City 2']-g['City 1']).dropna().unique())])

_merge
both          96
left_only      0
right_only     0
priced rows that equal the reference: 93 of 93

the 3 rows I left missing, with the reference value (not applied):
      date        item   city  price_ref
2026-01-01     Pulse C City 2      206.0
2026-01-15    Staple A City 2      109.0
2026-01-22 Sweetener D City 1      157.5

City 2 - City 1 gap in the cleaned data (all complete pairs): [6.0]
